# 📘 Notebook 02 · 量化策略 + 回测实战

> 多因子选股只是量化的一种。这一节我们走完剩下三种主流策略：CTA 趋势跟踪、统计套利、机器学习选股，并且建立**严格的样本外检验框架**。

**学完你会：**

- 实现并回测一个完整的 CTA 趋势跟踪策略
- 理解配对交易 + 协整检验
- 用 LightGBM 做截面选股，并用 SHAP 解释
- 掌握 walk-forward、purged k-fold 这些真正能防过拟合的方法
- 知道回测里那些"看起来漂亮但实际上是骗自己"的坑

**预计时间：10-15 小时**

## 1. CTA 趋势跟踪：最古老也最稳健的策略

### 1.1 思想

商品期货市场存在**趋势性**——一旦开始涨，就会继续涨一段时间；一旦开始跌，就会跌一段时间。原因可能是：

- **信息扩散慢**：基本面变化逐步被市场消化
- **从众行为**：投资者跟风加仓
- **风险溢价**：持有头寸需要补偿

CTA（Commodity Trading Advisor）就是利用这种规律的策略族。Renaissance、AQR、Man AHL 等顶级基金都有 CTA 产品。

### 1.2 三个经典 CTA 信号

| 信号 | 公式 | 意义 |
|---|---|---|
| **双均线** | $MA_{short} > MA_{long}$ → 多 | 经典中的经典，对小波段差 |
| **唐奇安通道** | 突破过去 N 日最高价 → 多 | 海龟交易系统核心 |
| **TSMOM** | 过去 12 月收益 > 0 → 多 | 学术经典 (Moskowitz et al. 2012) |

下面我们做唐奇安通道。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(2024)

# ---------- 生成一个商品期货价格 ----------
# 用 GBM + 偶发趋势制造一个有趋势的价格序列
n_days = 2000
dates = pd.bdate_range('2018-01-01', periods=n_days)

# 在某些时段嵌入趋势
trend = np.zeros(n_days)
trend[200:500]   =  0.0015    # 一段上涨趋势
trend[800:1100]  = -0.0012    # 一段下跌趋势
trend[1400:1700] =  0.0018    # 又一段上涨

# 日收益 = 趋势 + 噪声
ret = trend + np.random.randn(n_days) * 0.015
price = pd.Series(100 * (1 + ret).cumprod(), index=dates, name='price')

# 模拟 OHLC（用日收益加噪声构造）
high = price * (1 + np.abs(np.random.randn(n_days)) * 0.008)
low  = price * (1 - np.abs(np.random.randn(n_days)) * 0.008)
ohlc = pd.DataFrame({'open': price.shift(1).fillna(100),
                     'high': high, 'low': low, 'close': price})

fig, ax = plt.subplots(figsize=(11, 4))
ohlc['close'].plot(ax=ax, color='steelblue')
ax.set_title('模拟商品期货价格（嵌入了 3 段趋势）')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
def donchian_strategy(ohlc, entry_window=20, exit_window=10, atr_window=20, atr_mult=2):
    """
    海龟唐奇安通道策略
    - 突破过去 entry_window 日最高价 → 做多
    - 跌破过去 exit_window 日最低价 → 平多
    - 用 ATR 跟踪止损
    """
    df = ohlc.copy()

    # 唐奇安通道
    df['upper'] = df['high'].rolling(entry_window).max().shift(1)  # shift 防止用今天的高
    df['lower'] = df['low'].rolling(exit_window).min().shift(1)

    # ATR (Average True Range)
    tr = pd.concat([
        df['high'] - df['low'],
        (df['high'] - df['close'].shift()).abs(),
        (df['low'] - df['close'].shift()).abs()
    ], axis=1).max(axis=1)
    df['atr'] = tr.rolling(atr_window).mean()

    # 信号：先生成原始 entry / exit
    long_entry = df['close'] > df['upper']
    long_exit = df['close'] < df['lower']

    # 状态机：把信号转成持仓 (0/1)
    pos = np.zeros(len(df))
    stop_price = np.nan
    for i in range(1, len(df)):
        if pos[i-1] == 0:
            if long_entry.iloc[i]:
                pos[i] = 1
                stop_price = df['close'].iloc[i] - atr_mult * df['atr'].iloc[i]
            else:
                pos[i] = 0
        else:  # 持仓中
            # 更新跟踪止损（只上移不下移）
            stop_price = max(stop_price, df['close'].iloc[i] - atr_mult * df['atr'].iloc[i])
            if long_exit.iloc[i] or df['close'].iloc[i] < stop_price:
                pos[i] = 0
                stop_price = np.nan
            else:
                pos[i] = 1
    df['pos'] = pos

    # 策略收益 = 持仓 × 下一日价格变化 / 当前价
    df['ret'] = df['close'].pct_change()
    df['strategy_ret'] = df['pos'].shift(1) * df['ret']
    return df

result = donchian_strategy(ohlc, entry_window=20, exit_window=10, atr_mult=2)
print(result[['close', 'upper', 'lower', 'atr', 'pos', 'strategy_ret']].tail(10).round(3))

In [ ]:
# 评估
def quick_stats(returns, periods=252):
    cum = (1 + returns).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    ann_ret = (1 + returns.mean()) ** periods - 1
    ann_vol = returns.std() * np.sqrt(periods)
    return {
        '年化收益': ann_ret,
        '年化波动': ann_vol,
        '夏普    ': ann_ret / ann_vol if ann_vol > 0 else 0,
        '最大回撤': dd.min(),
        '胜率    ': (returns > 0).mean(),
        '交易次数': int((result['pos'].diff().abs() > 0).sum() / 2),
    }

stats = quick_stats(result['strategy_ret'].dropna())
print('唐奇安策略表现：')
for k, v in stats.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

# 画图
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax1 = axes[0]
ax1.plot(result.index, result['close'], color='gray', alpha=0.6, label='价格')
# 标出多头持仓期
in_position = result['pos'] == 1
ax1.fill_between(result.index, result['close'].min(), result['close'].max(),
                  where=in_position, alpha=0.2, color='green', label='多头持仓')
ax1.legend(); ax1.set_title('唐奇安策略持仓示意'); ax1.grid(alpha=0.3)

axes[1].plot(result.index, (1 + result['strategy_ret'].fillna(0)).cumprod(),
              color='darkred', label='策略净值')
axes[1].plot(result.index, (1 + result['ret'].fillna(0)).cumprod(),
              color='steelblue', alpha=0.7, label='买入持有')
axes[1].legend(); axes[1].set_title('策略 vs 买入持有'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**怎么看这个结果：**

- 趋势期内策略能抓住主升浪；震荡期内会有"假突破"的连续小亏
- 这就是 CTA 的本质：**靠少数大趋势赚钱，大部分时候亏小钱**
- 胜率通常 < 50%，但盈亏比 > 2，长期为正

**实战注意：**

1. 参数（entry_window、atr_mult）不要在历史数据上精调——很容易过拟合
2. 多品种组合：单品种 Sharpe 一般 0.5-1.0，10 个低相关品种组合能到 1.5-2.0
3. **大忌**：用每天的收盘价决策然后用收盘价成交——实际成交一般要顺延一天

### 🎯 练习

三个练习：

1. 把 entry_window 从 5 试到 60，画出 Sharpe 随参数变化的曲线。看到了什么？这告诉我们什么？
2. 实现 TSMOM 信号：如果过去 252 日（1 年）累计收益 > 0 → 持多；< 0 → 持空。对比 vs 唐奇安。
3. 这个回测里有一个潜在的未来函数——你能找出来吗？提示：和"假设我们能在收盘价瞬间成交"有关。

In [ ]:
# 在这里写你的答案


## 2. 配对交易：协整的经典应用

### 2.1 思想

两只**走势高度相关**的股票（如农行 / 工行，可口可乐 / 百事），价差应该围绕一个均值波动。当价差极端偏离时，**做多被低估的、做空被高估的**，等价差回归赚钱。

### 2.2 数学基础：协整 (Cointegration)

两个非平稳序列 $X_t, Y_t$ 如果存在 $\beta$ 使得 $X_t - \beta Y_t$ 是平稳的，则称它们**协整**。

这比"相关系数高"严格得多。两个 random walk 可能相关系数很高，但价差是发散的，不能套利。

**Engle-Granger 两步法：**
1. 用 OLS 估计：$X_t = \alpha + \beta Y_t + \epsilon_t$
2. 对残差 $\epsilon_t$ 做 ADF 检验，p < 0.05 说明残差平稳 → 协整成立

In [ ]:
# 制造一对协整的股票
np.random.seed(7)
common = np.cumsum(np.random.randn(500) * 0.5)   # 共同的随机游走
noise_a = np.random.randn(500) * 0.6
noise_b = np.random.randn(500) * 0.6

# A 和 B 共享一个共同因子，但有独立扰动
A = 50 + common + noise_a.cumsum() * 0.1
B = 30 + 0.6 * common + noise_b.cumsum() * 0.1

dates = pd.bdate_range('2022-01-01', periods=500)
pair = pd.DataFrame({'A': A, 'B': B}, index=dates)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pair.plot(ax=axes[0])
axes[0].set_title('两支模拟股票价格'); axes[0].grid(alpha=0.3)

# 计算价差
from statsmodels.api import OLS, add_constant
X = add_constant(pair['B'])
model = OLS(pair['A'], X).fit()
beta = model.params['B']
print(f'OLS 估计的 hedge ratio (β): {beta:.4f}')

spread = pair['A'] - beta * pair['B']
spread.plot(ax=axes[1], color='darkred')
axes[1].axhline(spread.mean(), color='black', linestyle='--', alpha=0.5)
axes[1].set_title(f'价差 spread = A - {beta:.2f}·B'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 协整检验
from statsmodels.tsa.stattools import coint, adfuller

# 方法 1: Engle-Granger 协整检验
score, pvalue, _ = coint(pair['A'], pair['B'])
print(f'Engle-Granger 协整检验：')
print(f'  统计量: {score:.4f}')
print(f'  p-value: {pvalue:.4f}  →  {"协整成立" if pvalue < 0.05 else "未发现协整"}')

# 方法 2: 对残差直接 ADF
adf_result = adfuller(spread)
print(f'\n残差 ADF 检验:')
print(f'  ADF 统计量: {adf_result[0]:.4f}')
print(f'  p-value: {adf_result[1]:.4f}')

In [ ]:
# 配对交易策略
def pair_trading(pair, lookback=60, entry_z=2.0, exit_z=0.5):
    """
    pair: DataFrame，两列分别是两支股票价格
    lookback: 滚动窗口用来算 hedge ratio 和 z-score
    """
    a = pair.iloc[:, 0]
    b = pair.iloc[:, 1]

    # 滚动估计 hedge ratio
    rolling_beta = (a.rolling(lookback).cov(b) / b.rolling(lookback).var())
    spread = a - rolling_beta * b

    # 滚动 z-score
    z = (spread - spread.rolling(lookback).mean()) / spread.rolling(lookback).std()

    # 信号
    long_signal  = z < -entry_z     # 价差太小 → 做多价差
    short_signal = z >  entry_z     # 价差太大 → 做空价差
    exit_signal  = z.abs() < exit_z

    # 状态机
    pos = 0
    positions = []
    for i in range(len(z)):
        if pd.isna(z.iloc[i]):
            positions.append(0); continue
        if pos == 0:
            if long_signal.iloc[i]: pos =  1
            elif short_signal.iloc[i]: pos = -1
        elif (pos == 1 and z.iloc[i] > -exit_z) or (pos == -1 and z.iloc[i] < exit_z):
            pos = 0
        positions.append(pos)

    df = pair.copy()
    df['spread'] = spread
    df['z'] = z
    df['pos'] = positions

    # 收益：做多价差 = +A收益 - β·B收益
    ret_a = a.pct_change()
    ret_b = b.pct_change()
    df['strategy_ret'] = df['pos'].shift(1) * (ret_a - rolling_beta.shift(1) * ret_b) / 2  # /2 因为多空各一半资金
    return df

bt = pair_trading(pair, lookback=60, entry_z=2.0, exit_z=0.5)

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
axes[0].plot(bt.index, bt['z'], color='steelblue', alpha=0.8)
axes[0].axhline( 2, color='red', linestyle='--', alpha=0.5)
axes[0].axhline(-2, color='red', linestyle='--', alpha=0.5)
axes[0].axhline(0, color='black', alpha=0.3)
axes[0].set_title('价差 Z-score 与持仓信号'); axes[0].grid(alpha=0.3)

axes[1].plot(bt.index, bt['pos'], color='darkred', drawstyle='steps')
axes[1].set_title('持仓 (1 做多价差, -1 做空价差, 0 空仓)'); axes[1].grid(alpha=0.3)

axes[2].plot(bt.index, (1 + bt['strategy_ret'].fillna(0)).cumprod(), color='darkgreen')
axes[2].set_title('策略净值'); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(quick_stats(bt['strategy_ret'].dropna()))

**配对交易常见坑：**

1. **协整关系会失效**：公司基本面变化、监管政策、行业格局都会让历史协整关系破裂。每隔一段时间重新检验是必须的。
2. **Hedge ratio 不稳定**：用滚动窗口动态更新，不要用固定值。
3. **流动性陷阱**：极端价差出现时往往伴随单边巨量，你想做空的那支可能根本融不到券。
4. **A 股做空成本高**：融券利率 6-10%，年化把多空利润吃掉一大半。

### 🎯 练习

1. 找两支真实的股票对（如同行业龙头），用 akshare 拉数据做协整检验
2. 测试 entry_z 从 1.5 到 3.0 对 Sharpe 的影响
3. 引入止损：如果 z 在某个方向上一直扩大超过 4，强制平仓认错

In [ ]:
# 在这里写你的答案


## 3. 机器学习选股：LightGBM 在量化中的标准用法

### 3.1 为什么不直接上深度学习

金融数据的特殊性：

- **信噪比极低**：日收益率几乎是白噪声，能解释方差 1-3% 就算"有信号"
- **样本量小**：5 年日频 = 1250 样本，深度网络容易过拟合
- **概念漂移**：分布会变化（市场状态切换）

**LightGBM / XGBoost 在多因子选股上是事实标准**，因为：

- 自动处理特征非线性、交互
- 对缺失值鲁棒
- 训练快、调参成熟
- 用 SHAP 可解释

### 3.2 一个完整 ML 选股流程

我们用 Notebook 01 的模拟数据，加上多个因子作为特征，预测下期收益排名。

In [ ]:
# 重新生成多因子数据（同 Notebook 01）
np.random.seed(42)
n_stocks, n_days = 200, 1500
dates = pd.bdate_range('2019-01-01', periods=n_days)
tickers = [f'S{i:03d}' for i in range(n_stocks)]

true_alpha = np.random.randn(n_stocks) * 0.0003
factor_loading = np.random.randn(n_stocks, 4) * 0.5
market = np.random.randn(n_days) * 0.012
style = np.random.randn(n_days, 4) * 0.008
beta_mkt = 0.8 + np.random.rand(n_stocks) * 0.6

returns = (true_alpha + np.outer(market, beta_mkt) + style @ factor_loading.T
           + np.random.randn(n_days, n_stocks) * 0.018)
returns = pd.DataFrame(returns, index=dates, columns=tickers)
price = 10 * (1 + returns).cumprod()
print('数据准备完毕:', price.shape)

In [ ]:
# ---------- 构造特征 ----------
def build_features(price, returns):
    feats = {}
    feats['mom_5']    = price.pct_change(5).shift(1)
    feats['mom_20']   = price.pct_change(20).shift(1)
    feats['mom_60']   = price.pct_change(60).shift(1)
    feats['rev_1']    = returns.shift(1) * -1
    feats['vol_20']   = returns.rolling(20).std().shift(1)
    feats['vol_60']   = returns.rolling(60).std().shift(1)
    feats['skew_20']  = returns.rolling(20).skew().shift(1)
    feats['kurt_20']  = returns.rolling(20).kurt().shift(1)
    feats['ma_ratio'] = (price.rolling(5).mean() / price.rolling(60).mean() - 1).shift(1)
    feats['high_close'] = (price.rolling(20).max() / price - 1).shift(1)
    return feats

features = build_features(price, returns)
print('特征数量:', len(features))
print('特征名称:', list(features.keys()))

In [ ]:
# ---------- 把特征 + 标签转成 long format 给 LightGBM ----------
def to_long_format(features_dict, label, min_date=None):
    """
    把 {feat_name: (date × stock)} 字典 + label (date × stock)
    转成 long format: 一行一个 (date, stock) 样本
    """
    long_dfs = []
    for name, f in features_dict.items():
        s = f.stack()
        s.name = name
        long_dfs.append(s)
    X = pd.concat(long_dfs, axis=1)
    y = label.stack()
    y.name = 'label'
    df = X.join(y, how='inner').dropna()
    df.index.names = ['date', 'stock']
    if min_date:
        df = df.loc[df.index.get_level_values('date') >= min_date]
    return df

# 标签：未来 5 日收益的截面排序（分类问题更稳定）
fwd_ret = price.pct_change(5).shift(-5)
# 把未来 5 日收益按日做横截面 rank（0~1），再二分类（前 30% 为 1，后 30% 为 0）
fwd_rank = fwd_ret.rank(axis=1, pct=True)
label = pd.DataFrame(np.nan, index=fwd_rank.index, columns=fwd_rank.columns)
label[fwd_rank > 0.7] = 1
label[fwd_rank < 0.3] = 0

dataset = to_long_format(features, label, min_date=pd.Timestamp('2019-04-01'))
print(f'样本数: {len(dataset):,}')
print(f'正样本比例: {(dataset["label"]==1).mean():.2%}')
dataset.head()

### 3.3 时间序列交叉验证（关键！）

**普通 KFold 在金融数据上是灾难**——它会让"未来的样本"出现在训练集里，导致回测虚高。

正确的做法：

1. **Walk-Forward / Expanding Window**：训练集只用 t 时刻之前的数据
2. **Purged K-Fold**（López de Prado）：训练集和测试集之间挖掉一段"清洗期"，避免标签重叠造成的泄漏

我们用 sklearn 的 `TimeSeriesSplit` 加 gap 参数实现 purged 版本。

In [ ]:
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print('未安装 lightgbm，跳过本节')
    print('安装: pip install lightgbm')

if HAS_LGB:
    from sklearn.model_selection import TimeSeriesSplit
    from sklearn.metrics import roc_auc_score

    feat_cols = list(features.keys())
    dataset = dataset.sort_index(level='date')

    # 按时间分割：前 70% 训练，30% 测试
    all_dates = dataset.index.get_level_values('date').unique().sort_values()
    split_idx = int(len(all_dates) * 0.7)
    split_date = all_dates[split_idx]

    train = dataset.loc[dataset.index.get_level_values('date') < split_date]
    test  = dataset.loc[dataset.index.get_level_values('date') >= split_date]

    X_tr, y_tr = train[feat_cols], train['label']
    X_te, y_te = test[feat_cols], test['label']

    model = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=5,
        min_child_samples=200,
        reg_lambda=0.1,
        verbose=-1,
    )
    model.fit(X_tr, y_tr)

    pred_te = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, pred_te)
    print(f'测试集 AUC: {auc:.4f}')
    print(f'(>0.55 算有效, 0.5 是随机)')

In [ ]:
if HAS_LGB:
    # ---------- 特征重要性 + SHAP ----------
    imp = pd.Series(model.feature_importances_, index=feat_cols).sort_values()

    fig, ax = plt.subplots(figsize=(8, 4))
    imp.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('LightGBM 特征重要性 (gain)')
    plt.tight_layout(); plt.show()

    # SHAP 值（如果装了）
    try:
        import shap
        explainer = shap.TreeExplainer(model)
        # 抽 1000 个样本看 SHAP
        sample = X_te.sample(min(1000, len(X_te)), random_state=42)
        sv = explainer.shap_values(sample)
        if isinstance(sv, list): sv = sv[1]    # 二分类正类
        shap_imp = pd.Series(np.abs(sv).mean(axis=0), index=feat_cols).sort_values()
        print('\nSHAP 平均绝对值（更稳定的特征重要性）:')
        print(shap_imp.round(4))
    except ImportError:
        print('未安装 shap，跳过 SHAP 分析')

In [ ]:
if HAS_LGB:
    # ---------- 把预测变成持仓做回测 ----------
    test = test.copy()
    test['pred'] = pred_te

    # 每天按预测排序，取前 20% 等权多头，后 20% 等权空头
    def daily_strategy(group):
        group = group.sort_values('pred', ascending=False)
        n_long = max(1, int(len(group) * 0.2))
        n_short = max(1, int(len(group) * 0.2))
        weights = pd.Series(0.0, index=group.index)
        weights.iloc[:n_long] =  1.0 / n_long
        weights.iloc[-n_short:] = -1.0 / n_short
        return weights

    # 注意：这里我们用 5 日未来收益作为标签，所以策略需要等持仓 5 天
    # 简化处理：每天用预测排序持仓 1 天（实战要重新设计）
    print('注：此处简化为日频持仓演示')

    # 看看预测概率分布
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(pred_te, bins=50, color='steelblue', alpha=0.7)
    ax.set_title('测试集预测概率分布'); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

**ML 选股实战的关键经验：**

1. **特征工程比模型选择重要 10 倍**——大多数 alpha 来自特征本身
2. **标签设计**：分类（前 N% vs 后 N%）通常比回归更稳健
3. **训练数据要够长**：至少 3 年滚动窗口
4. **重训频率**：月度或季度重训，不要每天重训（噪声主导）
5. **特征数 vs 样本数**：保守做法是特征数 < 样本数 / 50

**让面试官眼前一亮的细节：**

- 用 **SHAP 而不是 feature_importance**：后者对高基数变量有偏
- 主动提**标签重叠**（label overlap）问题：5 日未来收益作为标签，相邻样本的标签是相关的
- 知道 **purged group k-fold**：López de Prado《Advances in Financial ML》第 7 章

### 🎯 练习

1. 把标签改成"未来 5 日累计收益"做回归任务，比较分类 vs 回归的回测表现
2. 用 `TimeSeriesSplit(n_splits=5, gap=10)` 做 5 折 walk-forward CV，看每折的 AUC 稳定性
3. 加 5 个新特征（比如成交量类、量价相关性类），看 AUC 能不能提到 0.58 以上

In [ ]:
# 在这里写你的答案


## 4. 严格回测：怎么知道你的策略不是运气

### 4.1 回测的七宗罪

| 陷阱 | 表现 | 检测方法 |
|---|---|---|
| **未来函数** | 用了 t 之后才有的信息 | 把所有数据点的 `shift(1)` 检查一遍 |
| **幸存者偏差** | 只用现在还在的股票 | 必须用包含退市股的"全量历史" |
| **过拟合** | 参数微调能让 Sharpe 翻倍 | 看参数稳健性平面图 |
| **交易成本忽略** | 实盘比回测差 10x | 双向千三 + 万五滑点起步 |
| **流动性假设错** | 用收盘价瞬间成交 | 限制日内成交量 < 5% ADV |
| **PIT 问题** | 用了"修正后"的财报数据 | 检查数据时间戳 |
| **p-hacking** | 试 100 个挑 1 个 | 多重检验修正（Bonferroni、White Reality Check） |

### 4.2 Walk-forward 测试模板

In [ ]:
def walk_forward_backtest(features, label, model_class, model_kwargs,
                          train_window=252, test_window=63, step=63):
    """
    Walk-forward 回测：
    - 用前 train_window 天训练
    - 在接下来 test_window 天测试
    - 每 step 天向前滚动一次
    """
    if not HAS_LGB:
        print('需要 lightgbm');  return None

    # 准备 long format 数据
    feat_cols = list(features.keys())
    df = to_long_format(features, label, min_date=pd.Timestamp('2019-04-01'))
    df = df.sort_index(level='date')

    all_dates = df.index.get_level_values('date').unique().sort_values()
    results = []

    start = train_window
    while start + test_window <= len(all_dates):
        train_dates = all_dates[start - train_window : start]
        test_dates  = all_dates[start : start + test_window]

        train = df.loc[df.index.get_level_values('date').isin(train_dates)]
        test  = df.loc[df.index.get_level_values('date').isin(test_dates)]

        if len(train) < 100 or len(test) < 50:
            start += step; continue

        m = model_class(**model_kwargs)
        m.fit(train[feat_cols], train['label'])
        pred = m.predict_proba(test[feat_cols])[:, 1]

        test_out = test.copy()
        test_out['pred'] = pred
        from sklearn.metrics import roc_auc_score
        try:
            auc = roc_auc_score(test_out['label'], pred)
        except ValueError:
            auc = np.nan
        results.append({
            'test_start': test_dates[0],
            'test_end':   test_dates[-1],
            'auc':        auc,
            'n_train':    len(train),
            'n_test':     len(test),
        })
        start += step

    return pd.DataFrame(results)

if HAS_LGB:
    wf = walk_forward_backtest(
        features, label,
        lgb.LGBMClassifier,
        dict(n_estimators=200, learning_rate=0.05, num_leaves=31,
             max_depth=5, min_child_samples=200, verbose=-1),
        train_window=252, test_window=63, step=63
    )
    print('Walk-Forward 各折 AUC:')
    print(wf[['test_start', 'test_end', 'auc']].round(4))
    print(f'\nAUC 均值: {wf["auc"].mean():.4f}  标准差: {wf["auc"].std():.4f}'
          f'  > 0.5 的折数: {(wf["auc"]>0.5).sum()}/{len(wf)}')

### 4.3 多重检验修正

**问题场景：** 你试了 100 个因子组合，挑出 5 个 Sharpe > 1 的策略上线。这些策略真的有效吗？

**统计直觉：** 如果策略真实 Sharpe = 0，每个策略有 5% 概率"看起来"显著。100 个里就有期望 5 个假阳性。

**修正方法：**

- **Bonferroni**：把显著性阈值除以试验次数（如 0.05 / 100 = 0.0005）。保守但简单。
- **White Reality Check / Hansen SPA**：考虑多个策略的联合分布，p 值更准。
- **Block Bootstrap**：保留时间序列结构的抽样，估计原假设下的 Sharpe 分布。

下面用 block bootstrap 演示。

In [ ]:
def block_bootstrap_pvalue(strategy_returns, n_boot=1000, block_size=20):
    """
    用 block bootstrap 估计：在"真实期望收益=0"的原假设下，
    观察到当前夏普及以上的概率。
    block_size 保留时间序列依赖。
    """
    r = strategy_returns.dropna().values
    n = len(r)
    # 去掉均值（模拟原假设）
    r0 = r - r.mean()

    obs_sharpe = r.mean() / r.std() * np.sqrt(252)
    boot_sharpes = []
    n_blocks = n // block_size + 1
    for _ in range(n_boot):
        # 随机抽 block
        idx = np.random.randint(0, n - block_size, size=n_blocks)
        sample = np.concatenate([r0[i:i+block_size] for i in idx])[:n]
        boot_sharpes.append(sample.mean() / sample.std() * np.sqrt(252))
    boot_sharpes = np.array(boot_sharpes)
    pvalue = (boot_sharpes >= obs_sharpe).mean()
    return obs_sharpe, pvalue, boot_sharpes

# 用 CTA 策略测试
obs, pv, boots = block_bootstrap_pvalue(result['strategy_ret'].dropna(), n_boot=500, block_size=20)
print(f'观察到的 Sharpe : {obs:.3f}')
print(f'Bootstrap p 值 : {pv:.4f}  →  {"显著" if pv < 0.05 else "不显著"}')

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(boots, bins=40, color='lightgray', edgecolor='black', alpha=0.7)
ax.axvline(obs, color='red', linewidth=2, label=f'观察 Sharpe={obs:.3f}')
ax.set_title('Bootstrap 下的 Sharpe 分布（原假设：期望收益=0）')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. 实盘前最后检查清单

每个研究员入职后导师都会让他熟悉一遍这个清单：

### 数据
- [ ] 是否用了 PIT 数据（财报披露当日可得的版本）？
- [ ] 是否覆盖了退市股、停牌股、ST 股？
- [ ] 复权方式是否一致（前复权/后复权）？
- [ ] 时区、日期对齐是否正确？

### 策略
- [ ] 所有"特征"的 shift(1) 是否到位？
- [ ] 信号生成日和成交日是否分开？
- [ ] 是否考虑了涨跌停不能成交？
- [ ] 是否考虑了停牌？

### 成本
- [ ] 手续费：A 股双边万分之 12-25 + 千分之一印花税（卖出）
- [ ] 滑点：日频策略保守估计万分之五
- [ ] 冲击成本：单只股票日成交量 > 5% ADV 要加冲击成本模型
- [ ] 借券费：A 股做空考虑融券利率 6-10%

### 风险
- [ ] 单只股票仓位上限是否设？
- [ ] 行业敞口是否控制？
- [ ] 风格敞口（市值、价值）是否监控？
- [ ] 日内最大亏损止损？
- [ ] 黑天鹅情景测试（08 年金融危机、20 年熔断）？

### 工程
- [ ] 数据流断了怎么办？
- [ ] 下单失败重试逻辑？
- [ ] 报错告警发到哪？
- [ ] 每日 PnL 对账机制？

**如果你能对这些问题给出具体答案，面试官会觉得你像个真正的量化研究员，而不是 Kaggle 选手。**

## 6. 用专业框架：Backtrader 入门

自己写回测引擎适合学习，但生产中要用专业框架。**Backtrader** 是最容易上手的 Python 回测框架。

### 6.1 三大概念

- **Cerebro**：回测引擎主体
- **Strategy**：策略类，继承自 `bt.Strategy`
- **DataFeed**：数据源（OHLCV）

下面给一个最小的 Backtrader 策略示例。

In [ ]:
# 注：本 cell 在没装 backtrader 时只展示代码结构，不会真正运行
# 安装：pip install backtrader

backtrader_demo = '''
import backtrader as bt
import yfinance as yf

# ---------- 1. 定义策略 ----------
class SmaStrategy(bt.Strategy):
    params = (("fast", 10), ("slow", 30),)

    def __init__(self):
        self.sma_fast = bt.indicators.SMA(period=self.p.fast)
        self.sma_slow = bt.indicators.SMA(period=self.p.slow)
        self.crossover = bt.indicators.CrossOver(self.sma_fast, self.sma_slow)

    def next(self):
        if not self.position:                  # 没持仓
            if self.crossover > 0:             # 金叉
                self.buy()
        elif self.crossover < 0:               # 死叉
            self.close()

# ---------- 2. 准备数据 ----------
data = yf.download("AAPL", start="2020-01-01", end="2024-01-01")
data.columns = [c.lower() for c in data.columns]
data_feed = bt.feeds.PandasData(dataname=data)

# ---------- 3. 跑回测 ----------
cerebro = bt.Cerebro()
cerebro.adddata(data_feed)
cerebro.addstrategy(SmaStrategy, fast=10, slow=30)
cerebro.broker.set_cash(100000)
cerebro.broker.setcommission(commission=0.001)     # 千一手续费

cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name="sharpe")
cerebro.addanalyzer(bt.analyzers.DrawDown, _name="dd")

print(f"初始资金: {cerebro.broker.getvalue():.2f}")
results = cerebro.run()
print(f"最终资金: {cerebro.broker.getvalue():.2f}")
print(f"Sharpe: {results[0].analyzers.sharpe.get_analysis()}")
print(f"最大回撤: {results[0].analyzers.dd.get_analysis()}")

cerebro.plot()
'''
print(backtrader_demo)

### 6.2 Backtrader vs Qlib vs vnpy 怎么选

| 框架 | 强项 | 适合 |
|---|---|---|
| **Backtrader** | 上手快、灵活、社区大 | 学习、单标的策略、CTA |
| **Qlib** (微软开源) | 多因子选股专业、有自带数据 | A股多因子研究 |
| **bt** | 配置式语法、向量化、快 | 多资产组合再平衡 |
| **vnpy** | 国内主流，能实盘 | 国内实盘对接（CTP/期货公司） |
| **zipline** | quantopian 用过的 | 国外股票、教学 |

**入门推荐 Backtrader**，实盘推荐 **vnpy**（国内）或 **QuantConnect Lean**（国外）。

## 7. 本节小结

**你现在应该会的：**

- ✅ 实现 CTA 趋势跟踪（含 ATR 止损、跟踪止损）
- ✅ 配对交易：协整检验 + Z-score 策略
- ✅ 用 LightGBM 做机器学习选股
- ✅ Walk-Forward 时间序列交叉验证
- ✅ Block bootstrap 显著性检验
- ✅ 知道实盘前必查的所有清单项
- ✅ 用 Backtrader 写一个完整可跑的策略

**接下来：`03_量化企业实务_求职.ipynb`**

我们讲：私募/对冲基金内部怎么运转、岗位地图、面试常见题、简历怎么写、实习渠道。